Import Library

In [5]:
import pandas as pd
import requests
from datetime import timedelta
import time  

Configuration

In [6]:
API_URL = "https://api-sp2kp.kemendag.go.id/report/api/average-price/generate-perbandingan-harga"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

Parameters

In [7]:
start_date = "2026-01-01" 
end_date = "2026-04-17"

date_range = pd.date_range(start=start_date, end=end_date)

Extraction Loop

In [8]:
data_frames = []

for current_date in date_range:
    current_date_str = current_date.strftime("%Y-%m-%d")
    previous_date = current_date - timedelta(days=1)
    previous_date_str = previous_date.strftime("%Y-%m-%d")
    
    payload = {
        "tanggal": current_date_str,
        "tanggal_pembanding": previous_date_str
    }
    
    print(f"Fetching data for {current_date_str}\n", end=" ")
    
    try:
        response = requests.post(API_URL, headers=HEADERS, data=payload)
        
        if response.status_code == 200:
            json_data = response.json()
            raw_data = json_data.get("data", [])
            
            if raw_data:
                daily_df = pd.DataFrame(raw_data)
                
                target_columns = ['variant_id', 'variant_nama', 'satuan_display', 'tanggal', 'harga']
                clean_df = daily_df[target_columns].copy()
                
                data_frames.append(clean_df)
            else:
                print("WARNING: Empty data array returned.")
                
        else:
            print(f"ERROR: HTTP Status {response.status_code}.")
            
    except Exception as e:
        print(f"ERROR: Exception occurred - {e}")
    
    # Rate limiting to prevent IP block
    time.sleep(2) 

Fetching data for 2026-01-01
 Fetching data for 2026-01-02
 Fetching data for 2026-01-03
 Fetching data for 2026-01-04
 Fetching data for 2026-01-05
 Fetching data for 2026-01-06
 Fetching data for 2026-01-07
 Fetching data for 2026-01-08
 Fetching data for 2026-01-09
 Fetching data for 2026-01-10
 Fetching data for 2026-01-11
 Fetching data for 2026-01-12
 Fetching data for 2026-01-13
 Fetching data for 2026-01-14
 Fetching data for 2026-01-15
 Fetching data for 2026-01-16
 Fetching data for 2026-01-17
 Fetching data for 2026-01-18
 Fetching data for 2026-01-19
 Fetching data for 2026-01-20
 Fetching data for 2026-01-21
 Fetching data for 2026-01-22
 Fetching data for 2026-01-23
 Fetching data for 2026-01-24
 Fetching data for 2026-01-25
 Fetching data for 2026-01-26
 Fetching data for 2026-01-27
 Fetching data for 2026-01-28
 Fetching data for 2026-01-29
 Fetching data for 2026-01-30
 Fetching data for 2026-01-31
 Fetching data for 2026-02-01
 Fetching data for 2026-02-02
 Fetching d

Consolidating extracted data

In [10]:
if data_frames:
    master_df = pd.concat(data_frames, ignore_index=True)
    
    output_filename = f"dataset{len(data_frames)}.csv"
    master_df.to_csv(output_filename, index=False)
    
    print(f"INFO: Pipeline execution finished. Total rows: {len(master_df)}.")
    print(f"INFO: Data successfully exported to {output_filename}.")
else:
    print("CRITICAL: Pipeline failed. No data collected.")

INFO: Pipeline execution finished. Total rows: 1696.
INFO: Data successfully exported to dataset106.csv.
